# Live Demo — Clean & Build Panel
### Session 5 · ExInt II · WU Vienna · SS 2026

**What we do in this notebook:**
1. Find the most recent pull folder automatically
2. Read and concatenate all parquet chunks
3. Inspect the raw combined data
4. Drop duplicates and rows missing panel identifiers
5. Convert numeric columns
6. Sort into a firm-year panel
7. Save as `data/processed/panel_clean.parquet`

This is the **mini version** of `02_clean.py` — same logic, using the 5-firm demo data from the pull notebook.

> **Reference project:** https://github.com/vkiefner/sme-intl

---
## Cell 1 — Imports

In [ ]:
from pathlib import Path
from datetime import datetime

import pandas as pd

print(f"pandas: {pd.__version__}")

---
## Cell 2 — Find the most recent pull folder

Folders are named `YYYY-MM-DD_HH-MM-SS` so alphabetical sort = chronological sort.  
We always pick the last one — no manual path editing needed.

In [ ]:
RAW_DIR  = Path("data") / "raw"
PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

# List all timestamped pull folders
folders = sorted([f for f in RAW_DIR.iterdir() if f.is_dir()])

print("Pull folders found:")
for f in folders:
    files = list(f.glob("fyear_*.parquet"))
    print(f"  {f.name}  ({len(files)} parquet files)")

# Pick the most recent
latest = folders[-1]
print(f"\nUsing: {latest.name}")

---
## Cell 3 — Read all parquet chunks

In [ ]:
parquet_files = sorted(latest.glob("fyear_*.parquet"))
print(f"Reading {len(parquet_files)} files...\n")

chunks = []
for f in parquet_files:
    chunk = pd.read_parquet(f)
    chunks.append(chunk)
    print(f"  {f.name:<25}  {len(chunk):>4} rows")

# Concatenate into one DataFrame
df = pd.concat(chunks, ignore_index=True)
print(f"\nCombined: {df.shape[0]} rows × {df.shape[1]} columns")

---
## Cell 4 — First look at the combined data

In [ ]:
df.head(10)

In [ ]:
# Data types overview
print(f"Shape: {df.shape}")
print(f"\nDtype counts:")
print(df.dtypes.value_counts())

In [ ]:
# Panel coverage — firms × years
df.groupby(["gvkey", "conm"])["fyear"].agg(["min", "max", "count"]).rename(
    columns={"min": "first_year", "max": "last_year", "count": "n_years"}
)

---
## Cell 5 — Check for duplicate rows

In [ ]:
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}")

# Also check: are there duplicate gvkey-fyear combinations?
n_key_dupes = df.duplicated(subset=["gvkey", "fyear"]).sum()
print(f"Duplicate gvkey-fyear pairs: {n_key_dupes}")

# Drop exact duplicates
df = df.drop_duplicates()
print(f"\nAfter dedup: {len(df)} rows")

---
## Cell 6 — Drop rows missing panel identifiers

Every row in a firm-year panel **must** have a firm identifier (`gvkey`) and a time identifier (`fyear`).  
Without both, we cannot place the observation in the panel.

In [ ]:
n_before = len(df)

df = df.dropna(subset=["gvkey", "fyear"])

n_dropped = n_before - len(df)
print(f"Dropped {n_dropped} rows missing gvkey or fyear")
print(f"Remaining: {len(df)} rows")

---
## Cell 7 — Convert object columns to numeric where possible

WRDS sometimes returns numeric fields as `object` (string) dtype.  
We use `pd.to_numeric(..., errors='coerce')` — this converts what it can and turns non-numeric values into `NaN` (never crashes).

In [ ]:
# Columns we know are strings — leave these alone
STRING_COLS = {
    "gvkey", "conm", "cusip", "isin", "sedol", "tic",
    "naics", "sic", "loc", "curcd", "fic", "exchg",
    "costat", "stalt", "datafmt", "indfmt", "popsrc", "consol"
}

converted = []
for col in df.columns:
    if df[col].dtype == object and col not in STRING_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        converted.append(col)

print(f"Converted {len(converted)} columns to numeric dtype")
if converted:
    print("Columns converted:", converted)

---
## Cell 8 — Ensure `fyear` is integer

In [ ]:
df["fyear"] = df["fyear"].astype(int)
print(f"fyear dtype: {df['fyear'].dtype}")
print(f"fyear range: {df['fyear'].min()} – {df['fyear'].max()}")

---
## Cell 9 — Sort into a firm-year panel

In [ ]:
df = df.sort_values(["gvkey", "fyear"]).reset_index(drop=True)

print("Sorted by gvkey, fyear. First 10 rows:")
df[["gvkey", "conm", "fyear", "loc", "at", "sale", "ib", "xrd", "emp"]].head(10)

---
## Cell 10 — Missing value summary

In [ ]:
# How complete is each column?
missing = pd.DataFrame({
    "missing_n":   df.isnull().sum(),
    "missing_pct": (df.isnull().sum() / len(df) * 100).round(1),
    "dtype":       df.dtypes
})

# Show columns with at least one missing value, sorted by % missing
missing[missing["missing_n"] > 0].sort_values("missing_pct", ascending=False)

In [ ]:
# How complete are the key variables students will use?
key_vars = ["at", "sale", "ib", "xrd", "emp", "dltt", "pifo"]
available_key = [v for v in key_vars if v in df.columns]

print("Completeness of key variables:")
for col in available_key:
    pct_complete = (df[col].notna().sum() / len(df) * 100)
    bar = "█" * int(pct_complete / 5)
    print(f"  {col:<6}  {pct_complete:>5.1f}%  {bar}")

---
## Cell 11 — Panel statistics

In [ ]:
print("="*45)
print("Panel Statistics")
print("="*45)
print(f"  Total firm-years:   {len(df):>8,}")
print(f"  Unique firms:       {df['gvkey'].nunique():>8,}")
print(f"  Years covered:      {df['fyear'].min()}–{df['fyear'].max()}")
print(f"  Countries:          {df['loc'].nunique():>8,}")
print(f"  Total columns:      {df.shape[1]:>8,}")

print("\n  Observations per year:")
for year, count in df.groupby("fyear").size().items():
    print(f"    {year}: {count:>4}")

---
## Cell 12 — Quick descriptive stats on key variables

In [ ]:
df[available_key].describe().round(2)

---
## Cell 13 — Save clean panel as parquet

In [ ]:
out_path = PROC_DIR / "panel_clean.parquet"

df.to_parquet(out_path, index=False)

size_mb = out_path.stat().st_size / 1_048_576
print(f"Saved: {out_path}")
print(f"Size:  {size_mb:.2f} MB")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

---
## Cell 14 — Verify: read back and confirm

In [ ]:
check = pd.read_parquet(out_path)

print(f"Read back: {check.shape[0]} rows × {check.shape[1]} columns")
print(f"dtypes preserved: {check.dtypes.value_counts().to_dict()}")
check[["gvkey", "conm", "fyear", "at", "sale", "ib"]].head(10)

---
## Summary — What just happened

| Step | What we did | Why |
|------|-------------|-----|
| Found latest folder | `sorted(iterdir())[-1]` | No manual path editing — reproducible |
| Read chunks | `pd.read_parquet()` per file | Fast, type-safe |
| Concatenated | `pd.concat()` | One DataFrame for the whole panel |
| Dropped duplicates | `drop_duplicates()` | Avoid inflated row counts |
| Dropped missing IDs | `dropna(subset=[gvkey, fyear])` | Panel must be identifiable |
| Converted dtypes | `pd.to_numeric(..., errors='coerce')` | Numbers stored as numbers |
| Sorted | `sort_values([gvkey, fyear])` | Standard panel structure |
| Saved | `to_parquet()` | Compressed, fast, type-safe |

**Your full script `02_clean.py` does exactly this** — the only difference is it runs on all global firms, not just 5.

**Next session:** we use this panel in `03_descriptives.py` to build summary statistics and figures.

---
*ExInt II · WU Vienna · SS 2026 · github.com/vkiefner/sme-intl*